# 0. Introducción

En GeneReviews accedemos a los datos descargándolos directamente de la web. Están disponibles tanto en PDF como en HTML.
Dentro de algunas de las entradas se referencian varios genes que se deberían extraer y tratar por separado. Estas referencias están muy dispersas y no es posible extraerlas programáticamente.
Cuando existan varios genes que añada todos y marque el resto de parámetros con la mayor gravedad.

# Extracción

1. Se separa el texto útil de la sección de referencias y registro de modificaciones.
2. Se almacena en CSVs toda la información con las siguientes columnas:
    - Nombre del archivo -> Archivo
    - Nombre del documento -> Nombre
    - Titulo del documento -> Titulo
    - Fecha última modificación -> Fecha_Actualizacion
    - Contenido del documento -> Texto_Completo


Extaemos de un documento XML:
- El nombre del documento se encuentra en el árbol del XML en: object > book-part_wrapper > book_part > book_part_meta > book_part_id > __text
- El título del documento se encuentra en el árbol del XML en: object > book-part_wrapper > book_part > book_part_meta > title-group > title
- La fecha de la última actualización en el árbol del XML en: object > book-part_wrapper > book_part > book_part_meta > pub-history > date > "última entrada disponible". Formatea para que la fecha se registre con el formato DD-MM-YYYY
- Texto en el árbol del XML en: object > book-part_wrapper > book_part > body > sec > "todo el texto disponible"

In [ ]:
import xml.etree.ElementTree as ET
from datetime import datetime

def extraer_datos_xml(ruta_xml: str) -> dict: #in: ruta del xml, out: diccionario {nombre: str, titulo: str, fecha_actualizacion: str, texto: str}
    tree = ET.parse(ruta_xml)
    
    root = tree.getroot()

    # Navegación base
    base = root.find('book-part')
    #print(base.tag if base is not None else "No se encontró la etiqueta base")  # Verificar la etiqueta base

    # 1. Nombre del documento
    nombre_elem = base.find('book-part-meta/book-part-id') if base is not None else None
    nombre = nombre_elem.text if nombre_elem is not None else None

    # 2. Título del documento
    titulo_elem = base.find('book-part-meta/title-group/title') if base is not None else None
    titulo = titulo_elem.text if titulo_elem is not None else None

    # 3. Fecha última actualización
    fecha_formateada = None
    if base is not None:
        fechas = base.findall('book-part-meta/pub-history/date')
        #print(f"Fechas encontradas: {fechas}")  # Verificar las fechas encontradas

        if fechas:
            ultima_fecha = fechas[-1]  # última entrada disponible
            #print(ultima_fecha)

            dia = ultima_fecha.find('day').text
            mes = ultima_fecha.find('month').text
            anho = ultima_fecha.find('year').text

            fecha_formateada = dia + '-' + mes + '-' + anho


    # 4. XML completo dentro de <sec> (filtrando por título)
    secciones_xml = []
    titulos_excluidos = {"references", "chapter notes", "resources", "key sections in this"}

    if base is not None:
        secciones = base.findall('body/sec')
        #print(len(secciones))  # Verificar el número de secciones encontradas

        for sec in secciones:
            titulo_sec = sec.find('title')

            if titulo_sec is not None and titulo_sec.text:
                titulo_texto = titulo_sec.text.strip().lower()

                # Filtrar títulos no deseados
                #print(f"Procesando sección: '{titulo_texto}'")  # Verificar el título de cada sección
                if titulo_texto not in titulos_excluidos:
                    secciones_xml.append(ET.tostring(sec, encoding='unicode'))

    # Unir resultado
    xml_completo = "\n".join(secciones_xml)


    # Limpiar XML a texto plano (sin etiquetas ni saltos de línea)
    texto_limpio = []

    for sec_xml in secciones_xml:
        try:
            sec_root = ET.fromstring(sec_xml)
            texto = "".join(sec_root.itertext())

            # Eliminar saltos de línea y espacios extra
            texto = texto.replace("\n", " ").replace("\r", " ")
            texto = " ".join(texto.split())

            texto_limpio.append(texto.strip())
            
        except ET.ParseError:
            continue

    xml_completo = " ".join(texto_limpio)


    return {
        "nombre": nombre,
        "titulo": titulo,
        "fecha_actualizacion": fecha_formateada,
        "texto": xml_completo
    }

In [26]:

dict = extraer_datos_xml('Seleccion_GeneReviews/cf.nxml')
print(dict["nombre"])
print(dict["titulo"])
print(dict["fecha_actualizacion"])
print(dict["texto"][:])  # Imprime los primeros 100 caracteres del texto completo

cf
Cystic Fibrosis
8-8-2024
Diagnosis Consensus clinical diagnostic criteria for cystic fibrosis (CF) have been established [Farrell et al 2017a, Farrell et al 2017b, Sosnay et al 2017]. Suggestive Findings Scenario 1: Abnormal newborn screening (NBS) result NBS for CF is based on quantification of immunoreactive trypsinogen (IRT) and subsequent molecular testing including either CFTR targeted analysis or sequence analysis on dried blood spots. Elevated IRT values with or without the presence of CFTR pathogenic variants are considered out of range and require diagnostic sweat chloride testing. Scenario 2: Symptomatic individual not diagnosed during the newborn period. NBS was not universal in the United States until 2010; individuals not diagnosed following NBS may have: (1) been born prior to NBS implementation; (2) not had NBS for other reasons; (3) had a false negative NBS result; or (4) had caregivers who did not follow up with recommended diagnostic testing after abnormal NBS. Sug

In [ ]:
# Automatizamos la exteracción para la carpeta completa

import os

def procesar_carpeta(ruta_carpeta: str) -> list: #in: ruta de la carpeta, out: lista de diccionarios con los datos extraídos de cada archivo
    resultados = []

    for archivo in os.listdir(ruta_carpeta):
        if archivo.endswith(".nxml"):
            ruta_completa = os.path.join(ruta_carpeta, archivo)

            try:
                datos = extraer_datos_xml(ruta_completa)
                resultados.append({
                    "archivo": archivo,
                    "datos": datos
                })
            except Exception as e:
                resultados.append({
                    "archivo": archivo,
                    "error": str(e)
                })

    return resultados

In [29]:
resultados = procesar_carpeta('Seleccion_GeneReviews')

In [ ]:
# Los resultados obtenidos se guardan en un csv
import csv
def guardar_resultados_csv(resultados: list, ruta_csv: str) -> None: #in: lista de resultados, ruta del csv, out: None (guarda el csv Archivo | Nombre | Titulo | Fecha_Actualizacion | Texto_Completo)
    with open(ruta_csv, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Archivo", "Nombre", "Titulo", "Fecha_Actualizacion", "Texto_Completo"])

        for resultado in resultados:
            if "error" in resultado:
                writer.writerow([resultado["archivo"], "Error: " + resultado["error"], "", "", ""])
            else:
                datos = resultado["datos"]
                writer.writerow([
                    resultado["archivo"],
                    datos.get("nombre", ""),
                    datos.get("titulo", ""),
                    datos.get("fecha_actualizacion", ""),
                    datos.get("texto", "")
                ])
guardar_resultados_csv(resultados, 'resultados_gene_reviews.csv')

In [31]:
import pandas as pd

resultados_gene_reviews = pd.read_csv('resultados_gene_reviews.csv')

In [32]:
resultados_gene_reviews

,Archivo,Nombre,Titulo,Fecha_Actualizacion,Texto_Completo
0,cf.nxml,cf,Cystic Fibrosis,8-8-2024,Diagnosis Consensus clinical diagnostic criter...
1,cms.nxml,cms,Congenital Myasthenic Syndromes Overview,28-6-2012,1. Clinical Characteristics of Congenital Myas...
2,dcm-ov.nxml,dcm-ov,Dilated Cardiomyopathy Overview,12-12-2024,1. Dilated Cardiomyopathy (DCM): Definition Th...
3,deafness-overview.nxml,deafness-overview,Genetic Hearing Loss Overview,3-4-2025,1. Audiometric and Clinical Aspects of Hearing...
4,gaucher.nxml,gaucher,Gaucher Disease,7-12-2023,Diagnosis Suggestive Findings Scenario 1: Abno...
5,hht.nxml,hht,Hereditary Hemorrhagic Telangiectasia,16-3-2004,Diagnosis Consensus clinical diagnostic criter...
6,hyper-ftc.nxml,hyper-ftc,Hyperphosphatemic Familial Tumoral Calcinosis,1-2-2018,Diagnosis Suggestive Findings Hyperphosphatemi...
7,lchad.nxml,lchad,Long-Chain Hydroxyacyl-CoA Dehydrogenase Defic...,1-9-2022,Diagnosis No consensus clinical diagnostic cri...
8,perrault.nxml,perrault,Perrault Syndrome Overview,8-5-2025,1. Clinical Characteristics of Perrault Syndro...
9,pf.nxml,pf,Pulmonary Fibrosis Predisposition Overview,4-12-2025,1. Clinical Characteristics of Pulmonary Fibro...


In [33]:
nbk_omim = pd.read_csv('NBKid_shortname_OMIM.txt', sep='\t')

# añadimos el código NBK a una nueva columna en nuestro dataframe de resultados_gene_reviews usando el nombre del documento para hacer el match con el GR_shortname del nbk_omim
#resultados_gene_reviews = resultados_gene_reviews.merge(nbk_omim[['GR_shortname', '#NBK_id']], left_on='Nombre', right_on='GR_shortname', how='left')

resultados_gene_reviews = resultados_gene_reviews.merge(nbk_omim[['GR_shortname', '#NBK_id']].drop_duplicates(subset=['GR_shortname']), left_on='Nombre', right_on='GR_shortname', how='left')
resultados_gene_reviews = resultados_gene_reviews.rename(columns={'#NBK_id': 'NBK_id'})
resultados_gene_reviews = resultados_gene_reviews.drop(columns=['GR_shortname'])
resultados_gene_reviews.to_csv('resultados_gene_reviews.csv', index=False)
resultados_gene_reviews = pd.read_csv('resultados_gene_reviews.csv')
resultados_gene_reviews.head()


,Archivo,Nombre,Titulo,Fecha_Actualizacion,Texto_Completo,NBK_id
0,cf.nxml,cf,Cystic Fibrosis,8-8-2024,Diagnosis Consensus clinical diagnostic criter...,NBK1250
1,cms.nxml,cms,Congenital Myasthenic Syndromes Overview,28-6-2012,1. Clinical Characteristics of Congenital Myas...,NBK1168
2,dcm-ov.nxml,dcm-ov,Dilated Cardiomyopathy Overview,12-12-2024,1. Dilated Cardiomyopathy (DCM): Definition Th...,NBK1309
3,deafness-overview.nxml,deafness-overview,Genetic Hearing Loss Overview,3-4-2025,1. Audiometric and Clinical Aspects of Hearing...,NBK1434
4,gaucher.nxml,gaucher,Gaucher Disease,7-12-2023,Diagnosis Suggestive Findings Scenario 1: Abno...,NBK1269


# Exploramos csv que relacionan archivos de GeneReviews con otras páginas


In [34]:
import pandas as pd

nbk_omim = pd.read_csv('NBKid_shortname_OMIM.txt', sep='\t')
nbk_genesymbol = pd.read_csv('NBKid_shortname_genesymbol.txt', sep='\t')
nbk_shortname_title = pd.read_csv('GRtitle_shortname_NBKid.txt', sep='\t')
nbk_otros = pd.read_csv('GRshortname_NBKid_genesymbol_dzname.txt', sep='|', header=None, names=['mdel', 'NBK_id', 'genesymbol', 'dzname'])

# Cambiamos el nombre de las columnas para eliminar el "#"

nbk_omim.columns = [col.replace('#', '') for col in nbk_omim.columns]
nbk_genesymbol.columns = [col.replace('#', '') for col in nbk_genesymbol.columns]
nbk_shortname_title.columns = [col.replace('#', '') for col in nbk_shortname_title.columns]



In [35]:
# Contamos valores únicos de NBK_id en nbk_omim
print(nbk_omim["NBK_id"].nunique())

nbk_omim#.head() # columnas: NBK_id	GR_shortname	OMIM


880


,NBK_id,GR_shortname,OMIM
0,NBK1103,trimethylaminuria,136132
1,NBK1103,trimethylaminuria,602079
2,NBK1104,cdls,122470
3,NBK1104,cdls,300040
4,NBK1104,cdls,300269
...,...,...,...
4139,NBK621564,temple,616222
4140,NBK621565,fucosidosis,230000
4141,NBK621565,fucosidosis,612280
4142,NBK621569,kif1a-ndd,601255


In [36]:
# Contamos valores únicos de NBK_id en nbk_omim
print(nbk_shortname_title["NBK_id"].nunique())

nbk_shortname_title#.head() # columnas: GR_shortname	GR_Title NBK_id PMID


882


,GR_shortname,GR_Title,NBK_id,PMID
0,aadc-def,Aromatic L-Amino Acid Decarboxylase Deficiency,NBK595821,37824694.0
1,aars2-dis,AARS2-Related Disorder,NBK608563,39480987.0
2,ab-lipo-p,Abetalipoproteinemia,NBK532447,30358967.0
3,abs,Cytochrome P450 Oxidoreductase Deficiency,NBK1419,20301592.0
4,accpn,Hereditary Motor and Sensory Neuropathy with A...,NBK1372,20301546.0
...,...,...,...,...
877,yars1-def,YARS1 Deficiency,NBK615917,40608960.0
878,yci,Y Chromosome Infertility,NBK1339,20301513.0
879,yif1b-ndd,YIF1B-Related Neurodevelopmental Disorder,NBK606999,39265055.0
880,zap70-scid,ZAP70-Related Combined Immunodeficiency,NBK20221,20301777.0


In [37]:
nbk_shortname_title.query("NBK_id == 'NBK65707'")

,GR_shortname,GR_Title,NBK_id,PMID
622,pha2,Pseudohypoaldosteronism Type II,NBK65707,22073419.0


In [ ]:
df = nbk_omim.merge(nbk_shortname_title, on=["NBK_id", "GR_shortname"], how="left")

df_grouped = (
    df.groupby(["NBK_id", "GR_shortname", "GR_Title"])["OMIM"].apply(list).reset_index()
)

df_grouped["OMIM"] = df_grouped["OMIM"].apply(lambda x: list(set(x)))



In [39]:
print(df_grouped["NBK_id"].nunique())

df_grouped

880


,NBK_id,GR_shortname,GR_Title,OMIM
0,NBK100238,aprt-def,Adenine Phosphoribosyltransferase Deficiency,"[102600, 614723]"
1,NBK100239,hdls,CSF1R-Related Disorder,"[164770, 221820]"
2,NBK100240,pitt-hopkins,Pitt-Hopkins Syndrome,"[602272, 610954]"
3,NBK100241,hmdpc,Hypermanganesemia with Dystonia 1,"[613280, 611146]"
4,NBK100664,df-lamm,"Congenital Deafness with Labyrinthine Aplasia,...","[610706, 164950]"
...,...,...,...,...
875,NBK99167,pol3-leuk,POLR3-Related Leukodystrophy,"[610060, 614381, 607694, 616494, 614258, 614366]"
876,NBK99168,caffey,Caffey Disease,"[114000, 120150]"
877,NBK99494,dent,Dent Disease,"[300008, 300009, 300555, 300535]"
878,NBK99495,proteus,Proteus Syndrome,"[176920, 164730]"


In [40]:
df_grouped.to_parquet("nbk_omim.parquet", index=False)


In [41]:
df_grouped 

,NBK_id,GR_shortname,GR_Title,OMIM
0,NBK100238,aprt-def,Adenine Phosphoribosyltransferase Deficiency,"[102600, 614723]"
1,NBK100239,hdls,CSF1R-Related Disorder,"[164770, 221820]"
2,NBK100240,pitt-hopkins,Pitt-Hopkins Syndrome,"[602272, 610954]"
3,NBK100241,hmdpc,Hypermanganesemia with Dystonia 1,"[613280, 611146]"
4,NBK100664,df-lamm,"Congenital Deafness with Labyrinthine Aplasia,...","[610706, 164950]"
...,...,...,...,...
875,NBK99167,pol3-leuk,POLR3-Related Leukodystrophy,"[610060, 614381, 607694, 616494, 614258, 614366]"
876,NBK99168,caffey,Caffey Disease,"[114000, 120150]"
877,NBK99494,dent,Dent Disease,"[300008, 300009, 300555, 300535]"
878,NBK99495,proteus,Proteus Syndrome,"[176920, 164730]"


# Filtrado de los números OMIM
Algunos de los OMIM que aparecen relacionados con los NBK no tienen una relación directa con estos. Para ello se eliminarán los OMIM que tengan pocas menciones sobre la enfermedad.

In [12]:
import pandas as pd
df_grouped = pd.read_parquet("nbk_omim.parquet")
df_grouped

,NBK_id,GR_shortname,GR_Title,OMIM
0,NBK100238,aprt-def,Adenine Phosphoribosyltransferase Deficiency,"[102600, 614723]"
1,NBK100239,hdls,CSF1R-Related Disorder,"[164770, 221820]"
2,NBK100240,pitt-hopkins,Pitt-Hopkins Syndrome,"[602272, 610954]"
3,NBK100241,hmdpc,Hypermanganesemia with Dystonia 1,"[613280, 611146]"
4,NBK100664,df-lamm,"Congenital Deafness with Labyrinthine Aplasia,...","[610706, 164950]"
...,...,...,...,...
875,NBK99167,pol3-leuk,POLR3-Related Leukodystrophy,"[610060, 614381, 607694, 616494, 614258, 614366]"
876,NBK99168,caffey,Caffey Disease,"[114000, 120150]"
877,NBK99494,dent,Dent Disease,"[300008, 300009, 300555, 300535]"
878,NBK99495,proteus,Proteus Syndrome,"[176920, 164730]"


In [13]:
resultados_gene_reviews = pd.read_csv('resultados_gene_reviews.csv')
resultados_gene_reviews 

,Archivo,Nombre,Titulo,Fecha_Actualizacion,Texto_Completo,NBK_id
0,cf.nxml,cf,Cystic Fibrosis,8-8-2024,Diagnosis Consensus clinical diagnostic criter...,NBK1250
1,cms.nxml,cms,Congenital Myasthenic Syndromes Overview,28-6-2012,1. Clinical Characteristics of Congenital Myas...,NBK1168
2,dcm-ov.nxml,dcm-ov,Dilated Cardiomyopathy Overview,12-12-2024,1. Dilated Cardiomyopathy (DCM): Definition Th...,NBK1309
3,deafness-overview.nxml,deafness-overview,Genetic Hearing Loss Overview,3-4-2025,1. Audiometric and Clinical Aspects of Hearing...,NBK1434
4,gaucher.nxml,gaucher,Gaucher Disease,7-12-2023,Diagnosis Suggestive Findings Scenario 1: Abno...,NBK1269
5,hht.nxml,hht,Hereditary Hemorrhagic Telangiectasia,16-3-2004,Diagnosis Consensus clinical diagnostic criter...,NBK1351
6,hyper-ftc.nxml,hyper-ftc,Hyperphosphatemic Familial Tumoral Calcinosis,1-2-2018,Diagnosis Suggestive Findings Hyperphosphatemi...,NBK476672
7,lchad.nxml,lchad,Long-Chain Hydroxyacyl-CoA Dehydrogenase Defic...,1-9-2022,Diagnosis No consensus clinical diagnostic cri...,NBK583531
8,perrault.nxml,perrault,Perrault Syndrome Overview,8-5-2025,1. Clinical Characteristics of Perrault Syndro...,NBK242617
9,pf.nxml,pf,Pulmonary Fibrosis Predisposition Overview,4-12-2025,1. Clinical Characteristics of Pulmonary Fibro...,NBK1230


In [14]:
resultados_gene_reviews = resultados_gene_reviews.merge(df_grouped[['NBK_id', 'OMIM']], how='left', left_on='NBK_id', right_on='NBK_id')
resultados_gene_reviews.head()

,Archivo,Nombre,Titulo,Fecha_Actualizacion,Texto_Completo,NBK_id,OMIM
0,cf.nxml,cf,Cystic Fibrosis,8-8-2024,Diagnosis Consensus clinical diagnostic criter...,NBK1250,"[219700, 602421]"
1,cms.nxml,cms,Congenital Myasthenic Syndromes Overview,28-6-2012,1. Clinical Characteristics of Congenital Myas...,NBK1168,"[616321, 254210, 612866, 616322, 616323, 61632..."
2,dcm-ov.nxml,dcm-ov,Dilated Cardiomyopathy Overview,12-12-2024,1. Dilated Cardiomyopathy (DCM): Definition Th...,NBK1309,"[115200, 604288, 613121, 613122, 613252, 10254..."
3,deafness-overview.nxml,deafness-overview,Genetic Hearing Loss Overview,3-4-2025,1. Audiometric and Clinical Aspects of Hearing...,NBK1434,"[601093, 607237, 160775, 300039, 602121, 60519..."
4,gaucher.nxml,gaucher,Gaucher Disease,7-12-2023,Diagnosis Suggestive Findings Scenario 1: Abno...,NBK1269,"[231000, 608013, 230800, 230900, 168600, 23100..."


In [15]:
resultados_gene_reviews.to_parquet('seleccion_gene_reviews.parquet', index=False)

In [ ]:
# Para cada NBK_id i para cada OMIM asociado comprobamos cuantas menciones hay en el texto de OMIM al GR_title asociado y algunas variantes del título (sin puntuación, sin espacios, etc.).
# La comprobación se hará en la web de OMIM, extrayendo el texto completo de cada OMIM y contando las menciones. Para esto se usará la librería requests para hacer las peticiones a la web de OMIM y BeautifulSoup para extraer el texto.
# La web es https://www.omim.org/entry/OMIM_ID, donde OMIM_ID es el número de OMIM asociado a cada NBK_id.
# Si el número de menciones es menor que un umbral (por ejemplo, 5) se considerará que el OMIM no menciona el GR_title asociado y se eliminará de la lista de OMIM asociados a ese NBK_id.

import requests
from bs4 import BeautifulSoup
import re


def contar_menciones_omim(omim_id: str, gr_title: str) -> int: # in: omim_id, gr_title, out: número de menciones del gr_title y variantes en el texto de OMIM

    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0 Safari/537.36"
              }

    url = f"https://www.omim.org/entry/{omim_id}"
    print(f"Accediendo a {url} para contar menciones de '{gr_title}'...")
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, "html.parser")
        texto_omim = soup.get_text().lower()

        # Contar menciones del GR_title y variantes
        gr_title_variantes = set([
                                gr_title.lower(),
                                re.sub(r'[^\w\s]', '', gr_title.lower()),  # Sin puntuación
                                re.sub(r'\s+', '', gr_title.lower())  # Sin espacios
                                 ] 
                                 + gr_title.lower().split("-")[:-1]  # Palabras individuales del título sin el último guion
                                 + gr_title.lower().split(" ")[:-1])  # Palabras individuales del título
        print(gr_title_variantes) 

        conteo_menciones = sum(texto_omim.count(variante) for variante in gr_title_variantes)
        return conteo_menciones
    else:
        print(f"Error al acceder a OMIM ID {omim_id}: {response.status_code}")
        return 0


conteo = contar_menciones_omim("104000", "Adenine Phosphoribosyltransferase Deficiency")

Accediendo a https://www.omim.org/entry/104000 para contar menciones de 'Adenine Phosphoribosyltransferase Deficiency'...
{'adenine phosphoribosyltransferase deficiency', 'phosphoribosyltransferase', 'adenine', 'adeninephosphoribosyltransferasedeficiency'}


In [16]:
seleccion_genereviews = pd.read_parquet('seleccion_gene_reviews.parquet')

In [ ]:
import numpy as np
import time

def filtrar_omim_por_menciones(df: pd.DataFrame, umbral=15) -> pd.DataFrame: # in: dataframe con NBK_id, GR_shortname, GR_Title y lista de OMIM asociados, umbral de menciones, out: dataframe filtrado con solo los OMIM que mencionan el GR_Title asociado al menos umbral veces
    df_filtrado = df.copy()

    for index, row in df_filtrado.iterrows():
        nbk_id = row['NBK_id']
        omim_list = row['OMIM']
        #print(f"Procesando NBK_id {nbk_id} con OMIM asociados:")
        #print(type(omim_list))


        if isinstance(omim_list, (list, np.ndarray)) :
            gr_title = row['Titulo']  # Usamos el título del documento como GR_title asociado

            omim_filtrados = []
            for omim_id in omim_list:
                time.sleep(1)
                conteo_menciones = contar_menciones_omim(omim_id, gr_title)
                print(f"OMIM ID {omim_id} tiene {conteo_menciones} menciones de '{gr_title}'")

                if conteo_menciones >= umbral:
                    omim_filtrados.append(omim_id)

            df_filtrado.at[index, 'OMIM'] = omim_filtrados

    return df_filtrado

In [46]:
df = filtrar_omim_por_menciones(seleccion_genereviews, umbral=30)


Accediendo a https://www.omim.org/entry/219700 para contar menciones de 'Cystic Fibrosis'...
{'cystic', 'cysticfibrosis', 'cystic fibrosis'}
OMIM ID 219700 tiene 1644 menciones de 'Cystic Fibrosis'
Accediendo a https://www.omim.org/entry/602421 para contar menciones de 'Cystic Fibrosis'...
{'cystic', 'cysticfibrosis', 'cystic fibrosis'}
OMIM ID 602421 tiene 2140 menciones de 'Cystic Fibrosis'
Accediendo a https://www.omim.org/entry/616321 para contar menciones de 'Congenital Myasthenic Syndromes Overview'...
{'congenital myasthenic syndromes overview', 'congenitalmyasthenicsyndromesoverview', 'congenital', 'syndromes', 'myasthenic'}
OMIM ID 616321 tiene 103 menciones de 'Congenital Myasthenic Syndromes Overview'
Accediendo a https://www.omim.org/entry/254210 para contar menciones de 'Congenital Myasthenic Syndromes Overview'...
{'congenital myasthenic syndromes overview', 'congenitalmyasthenicsyndromesoverview', 'congenital', 'syndromes', 'myasthenic'}
OMIM ID 254210 tiene 153 mencione

KeyboardInterrupt: 